# Online Retail II: Monthly Churn Features and Labels

Build a complete customer-month panel from purchase transactions. Each month-end snapshot includes backward-looking 30-, 60-, and 90-day behavior, including months with no purchase. The target is no purchase in the 90 days after month-end. Only snapshots with a fully observed label window are retained.

## 1. Load cleaned data

In [2]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

DATA_PATH = '../data/outliers_handled.parquet'
transactions = pd.read_parquet(DATA_PATH)
required_columns = {
    'Invoice', 'StockCode', 'Quantity', 'InvoiceDate', 'Customer ID', 'Total_Price'
}
missing_columns = required_columns.difference(transactions.columns)
if missing_columns:
    raise ValueError(f'Missing required transaction columns: {sorted(missing_columns)}')

transactions['InvoiceDate'] = pd.to_datetime(transactions['InvoiceDate'])
transactions = transactions.dropna(subset=['Customer ID', 'InvoiceDate']).copy()
if 'Is_Cancellation' in transactions.columns:
    transactions = transactions[~transactions['Is_Cancellation'].astype(bool)].copy()
transactions = transactions[
    (transactions['Quantity'] > 0) & (transactions['Total_Price'] > 0)
].copy()
transactions['Customer ID'] = transactions['Customer ID'].astype(int)
transactions = transactions.sort_values('InvoiceDate').reset_index(drop=True)

WINDOW_DAYS = 90
dataset_end_exclusive = transactions['InvoiceDate'].max().normalize() + pd.Timedelta(days=1)
first_month = transactions['InvoiceDate'].min().to_period('M').to_timestamp()
last_month = transactions['InvoiceDate'].max().to_period('M').to_timestamp()
all_months = pd.date_range(first_month, last_month, freq='MS')
labelable_months = [
    month_start for month_start in all_months
    if month_start + pd.offsets.MonthBegin(1) + pd.Timedelta(days=WINDOW_DAYS)
    <= dataset_end_exclusive
]
first_purchase_dates = transactions.groupby('Customer ID')['InvoiceDate'].min()
first_purchase_months = first_purchase_dates.dt.to_period('M').dt.to_timestamp()

print(f'Purchase rows: {len(transactions):,}')
print(f"Customers: {transactions['Customer ID'].nunique():,}")
print(f"Observed data through: {transactions['InvoiceDate'].max()}")
print(f'Labelable month-end snapshots: {len(labelable_months)}')
print(f'First labelable month: {labelable_months[0]:%Y-%m}')
print(f'Last labelable month: {labelable_months[-1]:%Y-%m}')

Purchase rows: 776,591
Customers: 5,852
Observed data through: 2011-12-09 12:50:00
Labelable month-end snapshots: 21
First labelable month: 2009-12
Last labelable month: 2011-08


## 2. Create the complete customer-month snapshot panel

Include every customer from their first purchase through each labelable month-end. A customer remains in the panel during months with no transactions.

In [3]:
panel_parts = []
for month_start in labelable_months:
    snapshot_exclusive = month_start + pd.offsets.MonthBegin(1)
    eligible_customers = first_purchase_dates.index[
        first_purchase_dates < snapshot_exclusive
    ]
    panel_parts.append(pd.DataFrame({
        'Customer ID': eligible_customers,
        'CustomerMonth': month_start,
        'snapshot_exclusive': snapshot_exclusive,
    }))

panel = (
    pd.concat(panel_parts, ignore_index=True)
    .sort_values(['CustomerMonth', 'Customer ID'])
    .reset_index(drop=True)
)
if panel.duplicated(['Customer ID', 'CustomerMonth']).any():
    raise ValueError('Panel contains duplicate customer-month snapshots.')

print(f'Customer-month snapshots: {len(panel):,}')
print(f"Customers represented: {panel['Customer ID'].nunique():,}")
print(f"Snapshots: {panel['CustomerMonth'].min():%Y-%m} to {panel['CustomerMonth'].max():%Y-%m}")
panel.head()

Customer-month snapshots: 75,255
Customers represented: 5,224
Snapshots: 2009-12 to 2011-08


,Customer ID,CustomerMonth,snapshot_exclusive
0,12358,2009-12-01,2010-01-01
1,12359,2009-12-01,2010-01-01
2,12417,2009-12-01,2010-01-01
3,12422,2009-12-01,2010-01-01
4,12435,2009-12-01,2010-01-01


## 3. Add backward-looking behavior features

For each month-end, aggregate 30-, 60-, and 90-day order counts, spend, quantity, and unique products using transactions strictly before the next month starts. Add recency and short-term changes relative to the preceding 30 days.

In [6]:
panel = panel[['Customer ID', 'CustomerMonth', 'snapshot_exclusive']].copy()
feature_parts = []
window_days = (30, 60, 90)

for month_start in labelable_months:
    snapshot = panel[panel['CustomerMonth'].eq(month_start)].copy()
    snapshot_exclusive = month_start + pd.offsets.MonthBegin(1)

    for days in window_days:
        window_start = snapshot_exclusive - pd.Timedelta(days=days)
        window_transactions = transactions.loc[
            (transactions['InvoiceDate'] >= window_start)
            & (transactions['InvoiceDate'] < snapshot_exclusive)
        ]
        window_summary = (
            window_transactions.groupby('Customer ID')
            .agg(
                order_count=('Invoice', 'nunique'),
                spend=('Total_Price', 'sum'),
                quantity=('Quantity', 'sum'),
                unique_items=('StockCode', 'nunique'),
            )
            .rename(columns=lambda name: f'{name}_{days}d')
            .reset_index()
        )
        snapshot = snapshot.merge(window_summary, on='Customer ID', how='left')

    feature_parts.append(snapshot)

panel = pd.concat(feature_parts, ignore_index=True)
window_feature_cols = [
    f'{name}_{days}d'
    for days in window_days
    for name in ('order_count', 'spend', 'quantity', 'unique_items')
]
panel[window_feature_cols] = panel[window_feature_cols].fillna(0)

purchase_events = (
    transactions[['Customer ID', 'InvoiceDate']]
    .drop_duplicates()
    .sort_values('InvoiceDate')
)
anchors = panel[['Customer ID', 'CustomerMonth', 'snapshot_exclusive']].sort_values(
    'snapshot_exclusive'
)
anchors['snapshot_exclusive'] = anchors['snapshot_exclusive'].astype('datetime64[ns]')
purchase_events['InvoiceDate'] = purchase_events['InvoiceDate'].astype('datetime64[ns]')
last_purchase = pd.merge_asof(
    anchors,
    purchase_events,
    left_on='snapshot_exclusive',
    right_on='InvoiceDate',
    by='Customer ID',
    direction='backward',
    allow_exact_matches=False,
)
panel = panel.merge(
    last_purchase[['Customer ID', 'CustomerMonth', 'InvoiceDate']],
    on=['Customer ID', 'CustomerMonth'],
    how='left',
    validate='one_to_one',
).rename(columns={'InvoiceDate': 'last_purchase_date'})
panel['days_since_last_purchase'] = (
    panel['snapshot_exclusive'] - panel['last_purchase_date']
).dt.days
panel['is_first_month'] = panel['CustomerMonth'].eq(
    panel['Customer ID'].map(first_purchase_months)
)

panel['orders_previous_30d'] = panel['order_count_60d'] - panel['order_count_30d']
panel['spend_previous_30d'] = panel['spend_60d'] - panel['spend_30d']
panel['orders_change_30d'] = panel['order_count_30d'] - panel['orders_previous_30d']
panel['spend_change_30d'] = panel['spend_30d'] - panel['spend_previous_30d']
panel['spend_30d_to_previous_ratio'] = (
    panel['spend_30d'] / (panel['spend_previous_30d'] + 1)
)

panel['purchase_frequency'] = panel['order_count_90d']
panel['total_spend'] = panel['spend_90d']
panel['avg_order_value'] = np.divide(
    panel['spend_90d'],
    panel['order_count_90d'],
    out=np.zeros(len(panel), dtype=float),
    where=panel['order_count_90d'].to_numpy() > 0,
)
panel['unique_items'] = panel['unique_items_90d']
panel['has_purchase_30d'] = panel['order_count_30d'].gt(0).astype(int)
panel['has_purchase_90d'] = panel['order_count_90d'].gt(0).astype(int)

print(f'Panel with backward-looking features: {len(panel):,} rows')
print(f"Zero-purchase-in-30d snapshots: {panel['has_purchase_30d'].eq(0).sum():,}")
print(f"Missing recency values: {panel['days_since_last_purchase'].isna().sum():,}")
panel.head()

Panel with backward-looking features: 75,255 rows
Zero-purchase-in-30d snapshots: 54,888
Missing recency values: 0


,Customer ID,CustomerMonth,snapshot_exclusive,order_count_30d,spend_30d,quantity_30d,unique_items_30d,order_count_60d,spend_60d,quantity_60d,unique_items_60d,order_count_90d,spend_90d,quantity_90d,unique_items_90d,last_purchase_date,days_since_last_purchase,is_first_month,orders_previous_30d,spend_previous_30d,orders_change_30d,spend_change_30d,spend_30d_to_previous_ratio,purchase_frequency,total_spend,avg_order_value,unique_items,has_purchase_30d,has_purchase_90d
0,12358,2009-12-01,2010-01-01,1.0,1429.83,309.0,17.0,1.0,1429.83,309.0,17.0,1.0,1429.83,309.0,17.0,2009-12-08 07:59:00,23,True,0.0,0.0,1.0,1429.83,1429.83,1.0,1429.83,1429.830,17.0,1,1
1,12359,2009-12-01,2010-01-01,2.0,719.81,497.0,34.0,2.0,719.81,497.0,34.0,2.0,719.81,497.0,34.0,2009-12-16 15:24:00,15,True,0.0,0.0,2.0,719.81,719.81,2.0,719.81,359.905,34.0,1,1
2,12417,2009-12-01,2010-01-01,2.0,283.40,150.0,18.0,2.0,283.40,150.0,18.0,2.0,283.40,150.0,18.0,2009-12-10 11:25:00,21,True,0.0,0.0,2.0,283.40,283.40,2.0,283.40,141.700,18.0,1,1
3,12422,2009-12-01,2010-01-01,1.0,75.00,60.0,5.0,1.0,75.00,60.0,5.0,1.0,75.00,60.0,5.0,2009-12-18 12:44:00,13,True,0.0,0.0,1.0,75.00,75.00,1.0,75.00,75.000,5.0,1,1
4,12435,2009-12-01,2010-01-01,1.0,302.40,144.0,1.0,1.0,302.40,144.0,1.0,1.0,302.40,144.0,1.0,2009-12-02 13:58:00,29,True,0.0,0.0,1.0,302.40,302.40,1.0,302.40,302.400,1.0,1,1


## 4. Label the 90 days after each month-end snapshot

For each customer who has purchased by the snapshot, `is_churn = 1` when there is no purchase in the next 90 days. The prediction snapshot is month-end, features use transactions strictly before the next month starts, and snapshots without a fully observable 90-day window are excluded.

In [7]:
labeled_parts = []
for month_start in labelable_months:
    future_window_start = month_start + pd.offsets.MonthBegin(1)
    future_window_end = future_window_start + pd.Timedelta(days=WINDOW_DAYS)
    future_customers = transactions.loc[
        (transactions['InvoiceDate'] >= future_window_start)
        & (transactions['InvoiceDate'] < future_window_end),
        'Customer ID',
    ].unique()

    snapshot = panel.loc[panel['CustomerMonth'].eq(month_start)].copy()
    snapshot['label_window_start'] = future_window_start
    snapshot['label_window_end'] = future_window_end
    snapshot['is_churn'] = (
        ~snapshot['Customer ID'].isin(future_customers)
    ).astype(int)
    labeled_parts.append(snapshot)

panel = pd.concat(labeled_parts, ignore_index=True)
print('Label counts:')
print(panel['is_churn'].value_counts().rename(index={0: 'Retained', 1: 'Churn'}))
print(f"Overall churn rate: {panel['is_churn'].mean():.2%}")

Label counts:
is_churn
Churn       43541
Retained    31714
Name: count, dtype: int64
Overall churn rate: 57.86%


## 5. Confirm the label windows are observable

The snapshot list is filtered before feature and label construction so every retained month has the full 90-day future window inside the dataset.

In [8]:
panel['is_churn'] = panel['is_churn'].astype('int8')

assert panel['is_churn'].notna().all()
assert panel['label_window_start'].eq(panel['snapshot_exclusive']).all()
assert (panel['label_window_end'] - panel['label_window_start']).eq(
    pd.Timedelta(days=WINDOW_DAYS)
).all()
assert panel['label_window_end'].le(dataset_end_exclusive).all()
assert panel['Customer ID'].map(first_purchase_dates).lt(
    panel['snapshot_exclusive']
).all()
assert not panel.duplicated(['Customer ID', 'CustomerMonth']).any()

print(f"Rows: {len(panel):,}")
print(f"Customers: {panel['Customer ID'].nunique():,}")
print(f"Snapshot range: {panel['CustomerMonth'].min():%Y-%m} to {panel['CustomerMonth'].max():%Y-%m}")
print(f"Churn rate: {panel['is_churn'].mean():.2%}")
print('All snapshots have known labels and a fully observed 90-day window.')
panel.groupby('CustomerMonth')['is_churn'].mean().rename('churn_rate').to_frame()

Rows: 75,255
Customers: 5,224
Snapshot range: 2009-12 to 2011-08
Churn rate: 57.86%
All snapshots have known labels and a fully observed 90-day window.


,churn_rate
CustomerMonth,
2009-12-01,0.370137
2010-01-01,0.373768
2010-02-01,0.378394
2010-03-01,0.429977
2010-04-01,0.466447
2010-05-01,0.502981
2010-06-01,0.511352
2010-07-01,0.455676
2010-08-01,0.416793


## 6. Validate features and labels

- No missing or non-finite predictor values
- Each customer-month appears once and uses only purchases before the snapshot boundary
- The 90-day label interval is fully observable and matches the month-end definition
- A sampled rolling feature and churn label are independently checked against raw transactions

In [9]:
feature_columns = [
    column for column in panel.columns
    if column.endswith(('30d', '60d', '90d'))
] + [
    'days_since_last_purchase', 'is_first_month',
    'orders_previous_30d', 'spend_previous_30d',
    'orders_change_30d', 'spend_change_30d',
    'spend_30d_to_previous_ratio', 'purchase_frequency', 'total_spend',
    'avg_order_value', 'unique_items', 'has_purchase_30d', 'has_purchase_90d',
]
assert panel[feature_columns].notna().all().all()
assert np.isfinite(panel[feature_columns].select_dtypes(include='number')).all().all()

sample = panel.sample(n=1, random_state=42).iloc[0]
sample_customer = sample['Customer ID']
snapshot_boundary = sample['snapshot_exclusive']
for days in window_days:
    sample_window = transactions.loc[
        (transactions['Customer ID'] == sample_customer)
        & (transactions['InvoiceDate'] >= snapshot_boundary - pd.Timedelta(days=days))
        & (transactions['InvoiceDate'] < snapshot_boundary)
    ]
    assert sample[f'order_count_{days}d'] == sample_window['Invoice'].nunique()
    assert np.isclose(sample[f'spend_{days}d'], sample_window['Total_Price'].sum())

sample_future = transactions.loc[
    (transactions['Customer ID'] == sample_customer)
    & (transactions['InvoiceDate'] >= sample['label_window_start'])
    & (transactions['InvoiceDate'] < sample['label_window_end'])
]
expected_label = int(sample_future.empty)
assert sample['is_churn'] == expected_label

print(f'Feature columns checked: {len(feature_columns)}')
print('No missing/non-finite features; sampled rolling values and label match raw transactions.')

Feature columns checked: 31
No missing/non-finite features; sampled rolling values and label match raw transactions.


## 7. Assemble and save the modeling table

Save identifiers, backward-looking features, and the churn target. Snapshot and label-window helper timestamps are excluded so they cannot accidentally become predictors.

In [10]:
rolling_feature_columns = [
    f'{name}_{days}d'
    for days in window_days
    for name in ('order_count', 'spend', 'quantity', 'unique_items')
]
additional_feature_columns = [
    'days_since_last_purchase', 'is_first_month',
    'orders_previous_30d', 'spend_previous_30d',
    'orders_change_30d', 'spend_change_30d',
    'spend_30d_to_previous_ratio', 'purchase_frequency', 'total_spend',
    'avg_order_value', 'unique_items', 'has_purchase_30d', 'has_purchase_90d',
]
feature_columns = [
    'Customer ID', 'CustomerMonth',
    *rolling_feature_columns,
    *additional_feature_columns,
    'is_churn',
]
if len(feature_columns) != len(set(feature_columns)):
    raise ValueError('Feature table contains duplicate column names.')

customer_month_features = panel[feature_columns].reset_index(drop=True)
print('Final feature table shape:', customer_month_features.shape)
print('Predictors:', len(feature_columns) - 3)
print(customer_month_features.dtypes)
customer_month_features.head()

Final feature table shape: (75255, 28)
Predictors: 25
Customer ID                             int64
CustomerMonth                  datetime64[us]
order_count_30d                       float64
spend_30d                             float64
quantity_30d                          float64
unique_items_30d                      float64
order_count_60d                       float64
spend_60d                             float64
quantity_60d                          float64
unique_items_60d                      float64
order_count_90d                       float64
spend_90d                             float64
quantity_90d                          float64
unique_items_90d                      float64
days_since_last_purchase                int64
is_first_month                           bool
orders_previous_30d                   float64
spend_previous_30d                    float64
orders_change_30d                     float64
spend_change_30d                      float64
spend_30d_to_previous_rati

,Customer ID,CustomerMonth,order_count_30d,spend_30d,quantity_30d,unique_items_30d,order_count_60d,spend_60d,quantity_60d,unique_items_60d,order_count_90d,spend_90d,quantity_90d,unique_items_90d,days_since_last_purchase,is_first_month,orders_previous_30d,spend_previous_30d,orders_change_30d,spend_change_30d,spend_30d_to_previous_ratio,purchase_frequency,total_spend,avg_order_value,unique_items,has_purchase_30d,has_purchase_90d,is_churn
0,12358,2009-12-01,1.0,1429.83,309.0,17.0,1.0,1429.83,309.0,17.0,1.0,1429.83,309.0,17.0,23,True,0.0,0.0,1.0,1429.83,1429.83,1.0,1429.83,1429.830,17.0,1,1,1
1,12359,2009-12-01,2.0,719.81,497.0,34.0,2.0,719.81,497.0,34.0,2.0,719.81,497.0,34.0,15,True,0.0,0.0,2.0,719.81,719.81,2.0,719.81,359.905,34.0,1,1,0
2,12417,2009-12-01,2.0,283.40,150.0,18.0,2.0,283.40,150.0,18.0,2.0,283.40,150.0,18.0,21,True,0.0,0.0,2.0,283.40,283.40,2.0,283.40,141.700,18.0,1,1,0
3,12422,2009-12-01,1.0,75.00,60.0,5.0,1.0,75.00,60.0,5.0,1.0,75.00,60.0,5.0,13,True,0.0,0.0,1.0,75.00,75.00,1.0,75.00,75.000,5.0,1,1,0
4,12435,2009-12-01,1.0,302.40,144.0,1.0,1.0,302.40,144.0,1.0,1.0,302.40,144.0,1.0,29,True,0.0,0.0,1.0,302.40,302.40,1.0,302.40,302.400,1.0,1,1,0


In [11]:
OUTPUT_PATH = '../data/features_and_label.parquet'
customer_month_features.to_parquet(OUTPUT_PATH, index=False)

saved_features = pd.read_parquet(OUTPUT_PATH)
assert saved_features.shape == customer_month_features.shape
assert saved_features.columns.equals(customer_month_features.columns)
print(f'Saved and verified {saved_features.shape[0]:,} rows to {OUTPUT_PATH}')

Saved and verified 75,255 rows to ../data/features_and_label.parquet
